# Layer Importance Profiling

Which layers matter most? Comprehensive importance profiling via ablation,
gradient norms, output magnitudes, prediction impact, and cumulative buildup.

## Why This Matters

Layer importance characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_importance_profiling import (
    layer_ablation_importance, layer_gradient_importance,
    layer_output_magnitude, layer_prediction_impact,
    cumulative_layer_importance,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer Ablation Importance

Which layers cause the most damage when removed?

In [ ]:
result = layer_ablation_importance(model, tokens)
print(f"Most important: layer {result['most_important']}")
print(f"Least important: layer {result['least_important']}")
print(f"Critical layers: {result['n_critical']}\n")
for p in result['per_layer']:
    crit = 'CRITICAL' if p['is_critical'] else 'ok'
    print(f"  Layer {p['layer']}: KL={p['kl_divergence']:.4f}, "
          f"max_change={p['max_logit_change']:.4f} [{crit}]")

## Layer Gradient Importance

Residual stream norms show where computation accumulates.

In [ ]:
result = layer_gradient_importance(model, tokens, position=-1)
print(f"Position: {result['position']}, target token: {result['target_token']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['relative_norm'] * 30)
    print(f"  Layer {p['layer']}: norm={p['residual_norm']:.4f}, "
          f"relative={p['relative_norm']:.3f} {bar}")

## Layer Output Magnitude

Compare attention vs MLP output magnitudes at each layer.

In [ ]:
result = layer_output_magnitude(model, tokens)
print(f"Attn-dominated: {result['attn_dominated_layers']}, "
      f"MLP-dominated: {result['mlp_dominated_layers']}\n")
for p in result['per_layer']:
    dom = 'attn' if p['attn_fraction'] > 0.5 else 'MLP'
    print(f"  Layer {p['layer']}: attn={p['attn_norm']:.4f}, "
          f"mlp={p['mlp_norm']:.4f}, combined={p['combined_norm']:.4f} [{dom}]")

## Layer Prediction Impact

How much does each layer change the model's prediction?

In [ ]:
result = layer_prediction_impact(model, tokens)
print(f"Prediction changes: {result['n_prediction_changes']}")
print(f"Biggest shift: layer {result['biggest_shift_layer']}\n")
for p in result['per_layer']:
    changed = '← CHANGED' if p['prediction_changed'] else ''
    print(f"  Layer {p['layer']}: KL_from_prev={p['kl_from_previous']:.4f}, "
          f"top={p['top_token']} {changed}")

## Cumulative Layer Importance

How does the prediction build up through layers?

In [ ]:
result = cumulative_layer_importance(model, tokens)
print(f"Target token: {result['target_token']}")
print(f"Final projection: {result['final_projection']:.4f}\n")
for p in result['per_layer']:
    bar = '█' * int(abs(p['fraction_of_final']) * 30)
    print(f"  Layer {p['layer']}: cumulative={p['cumulative_projection']:.4f}, "
          f"increment={p['layer_increment']:+.4f}, frac={p['fraction_of_final']:.2f} {bar}")